# SMART Phase 1 -- QA / QC

Checks the published `Transportation_SMART` feature service after an ETL run.

Uses the same modules as the pipeline (`scripts/`), so the layer IDs, field
maps and query logic here cannot drift from what the ETL actually writes.

Run `python scripts/run_etl.py --preflight` first if you only want to confirm
the config still matches the live schemas -- that is faster than this notebook.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path().absolute().parent / "scripts"))

import transforms as T
from arcgis_client import ArcGISClient
from smart_config import load_settings

pd.options.display.max_columns = 999

settings = load_settings()
client = ArcGISClient(settings)
print(settings.service_url)

## 1. Row counts and date extents

The headline check after a run: did every layer grow, and how current is it?

In [ ]:
rows = []
for name, cfg in settings.datasets.items():
    date_field = cfg.get("date_field")
    latest = client.get_max_date(cfg["layer_id"], date_field) if date_field else None
    rows.append(
        {
            "dataset": name,
            "layer": cfg["layer_id"],
            "load_mode": cfg.get("load_mode"),
            "rows": client.count(cfg["layer_id"]),
            "date_field": date_field,
            "latest": latest.date() if latest else None,
            "days_behind": (pd.Timestamp.utcnow().date() - latest.date()).days
            if latest
            else None,
        }
    )

pd.DataFrame(rows)

## 2. Null checks on the fields the ETL writes

The original ETL would have written every `counts` and `Date` as null because
it never renamed DERQ's `count`/`date`. Any non-zero here means something in
the field mapping is wrong.

In [ ]:
checks = []
for name, cfg in settings.datasets.items():
    if cfg.get("derived_from"):
        fields = [cfg["sum_field"], *cfg["group_by"]]
    else:
        # Includes Camera_Status, which is enriched from the locations layer
        # rather than fetched from DERQ -- a spike in nulls there means the
        # join is failing, not that DERQ changed.
        fields = T.target_fields(cfg)
    total = client.count(cfg["layer_id"])
    for field in fields:
        nulls = client.count(cfg["layer_id"], f"{field} IS NULL")
        checks.append(
            {
                "dataset": name,
                "field": field,
                "nulls": nulls,
                "pct": round(100 * nulls / total, 2) if total else None,
            }
        )

nulls = pd.DataFrame(checks)
display(nulls[nulls["nulls"] > 0])
print("Fields fully populated:", int((nulls['nulls'] == 0).sum()), "of", len(nulls))

## 3. Duplicate-day check

`day_replace` should make re-runs idempotent. If a day's row count suddenly
doubles relative to its neighbours, the delete step is not matching the rows
the add step writes.

In [ ]:
daily = client.query_all(
    settings.dataset("daily_counts")["layer_id"],
    out_fields="Date,LocationName,counts",
)
daily["Date"] = pd.to_datetime(daily["Date"], unit="ms", utc=True).dt.date

per_day = daily.groupby(["Date", "LocationName"]).size().rename("rows").reset_index()
duplicates = per_day[per_day["rows"] > 1]
print(f"{len(duplicates)} (date, location) pair(s) with more than one row")
duplicates.head(20)

## 4. Daily totals reconcile against the 15-minute counts

Layer 5 is derived from layer 3, so the two must agree for any day present in
both. A mismatch means the derived layer is stale or double-loaded.

In [ ]:
# Pick a recent day that exists in both layers.
check_day = daily["Date"].max()
# Wide candidate window + exact epoch filter -- timestamp literals are
# compared in the database timezone, not UTC (see candidate_day_clause).
where = T.candidate_day_clause("Date", check_day)

fifteen_min = client.query_all(
    settings.dataset("vehicle_counts")["layer_id"],
    out_fields="Date,LocationName,counts",
    where=where,
)
lo, hi = T.epoch_day_bounds(check_day)
fifteen_min = fifteen_min[fifteen_min["Date"].between(lo, hi - 1)]

comparison = (
    fifteen_min.groupby("LocationName")["counts"].sum().rename("from_15min").to_frame()
    .join(
        daily[daily["Date"] == check_day]
        .set_index("LocationName")["counts"]
        .rename("from_daily"),
        how="outer",
    )
)
comparison["difference"] = comparison["from_daily"] - comparison["from_15min"]
print(f"Reconciling {check_day}")
comparison

## 5. Coverage by location

A camera that stops reporting shows up here as a stale `last_seen` rather than
as an error -- the DERQ API returns an empty body for a decommissioned sensor.

In [ ]:
coverage = (
    daily.groupby("LocationName")
    .agg(days=("Date", "nunique"), first_seen=("Date", "min"), last_seen=("Date", "max"))
    .sort_values("last_seen")
)

configured = set(client.get_locations()["LocationName"])
missing = configured - set(coverage.index)
if missing:
    print("Locations with no data at all:", sorted(missing))

coverage

## 6. Safety event mix

Sanity check on the event-type distribution. `STPV`, `TV` and `LCV` are still
unconfirmed expansions -- see `metadata/field_descriptions.yaml`.

In [ ]:
safety = client.query_all(
    settings.dataset("safety_insights")["layer_id"],
    out_fields="EventType,TimeAtSite,LocationName",
)
safety["TimeAtSite"] = pd.to_datetime(safety["TimeAtSite"], unit="ms", utc=True)

print(
    "Range:",
    safety['TimeAtSite'].min().date(),
    "to",
    safety['TimeAtSite'].max().date(),
)
safety["EventType"].value_counts().to_frame("events")

## 7. Location roster: DERQ vs the curated ArcGIS layer

The pipeline fetches data ONLY for cameras listed in the ArcGIS locations
layer (layer 0) -- that layer is the curated source of truth, so cameras are
added or retired by editing it, not the code.

The flip side: a camera that exists in DERQ but is missing from layer 0 is
**silently never fetched**. This section pulls DERQ's own `/locations` roster
(one API request) and diffs the two lists.

- **In DERQ only** -> add it to layer 0 (with Status and coordinates) if its
  data should be collected. Until then, none of its data reaches the service.
- **In ArcGIS only** -> the camera no longer exists in DERQ; every fetch for it
  returns empty. Its historical rows are untouched. Consider noting its
  DateRange on layer 0.
- **Name mismatches** matter because LocationName is denormalized onto every
  data row (and is the join key for the Daily Counts layer, which has no
  LocationId).

In [ ]:
from derq_client import DerqClient

derq_roster = DerqClient(settings).get_locations()
arcgis_roster = client.query_all(
    settings.locations["source_layer"],
    out_fields="LocationId,LocationName,Status,DateRange",
)

comparison = derq_roster.merge(
    arcgis_roster, on="LocationId", how="outer",
    suffixes=("_derq", "_arcgis"), indicator=True,
)

missing_from_arcgis = comparison[comparison["_merge"] == "left_only"]
missing_from_derq = comparison[comparison["_merge"] == "right_only"]
matched = comparison[comparison["_merge"] == "both"]
name_mismatch = matched[
    matched["LocationName_derq"].str.strip()
    != matched["LocationName_arcgis"].str.strip()
]

print(f"DERQ roster:   {len(derq_roster)} camera(s)")
print(f"ArcGIS layer:  {len(arcgis_roster)} camera(s)")
print(f"matched:       {len(matched)}")
print()

if missing_from_arcgis.empty:
    print("OK  every DERQ camera is in the ArcGIS layer")
else:
    print(f"!!  {len(missing_from_arcgis)} camera(s) in DERQ but NOT in layer 0 -- data not being fetched:")
    display(missing_from_arcgis[["LocationId", "LocationName_derq"]])

if missing_from_derq.empty:
    print("OK  every ArcGIS camera still exists in DERQ")
else:
    print(f"..  {len(missing_from_derq)} camera(s) in layer 0 but no longer in DERQ (fetches return empty):")
    display(missing_from_derq[["LocationId", "LocationName_arcgis", "Status", "DateRange"]])

if name_mismatch.empty:
    print("OK  no name mismatches")
else:
    print(f"!!  {len(name_mismatch)} name mismatch(es) -- LocationName is stamped on every data row:")
    display(name_mismatch[["LocationId", "LocationName_derq", "LocationName_arcgis"]])